In [66]:
import os
import json
import re
from pathlib import Path
from sentence_transformers import SentenceTransformer
from langchain_experimental.text_splitter import SemanticChunker
from PyPDF2 import PdfReader

In [67]:
import os

# Root of the project. Defaults to the current working directory,
# or override via the RAG_BASE_DIR environment variable.
BASE_DIR = Path(os.getenv("RAG_BASE_DIR", Path.cwd()))

EXTRACTED_DIR = BASE_DIR / "extracted"
CHUNKED_DIR   = BASE_DIR / "chunked"

PDF_DIR       = BASE_DIR / "raw_data"
BLOCK_DIR     = EXTRACTED_DIR / "blocks"
IMAGE_DIR     = EXTRACTED_DIR / "images"
TABLE_DIR     = EXTRACTED_DIR / "tables"

In [78]:
def load_page_components(pdf_name, page):
    page_str = f"{page:03d}"

    block_file = BLOCK_DIR / f"{pdf_name}_blocks_page_{page_str}.json"
    if block_file.exists():
        raw = json.load(open(block_file, "r", encoding="utf-8"))
        blocks = raw.get("blocks", [])
    else:
        blocks = []
    
    tables = []
    for file in TABLE_DIR.glob(f"{pdf_name}_page_{page_str}_table_*.json"):
        tables.append(json.load(open(file, "r", encoding="utf-8")))

    return blocks, tables

In [69]:
def clean_text(text):
    if not text:
        return ""
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [70]:
def merge_page_text(blocks):
    lines = []
    for block in blocks:
        if "text" in block:
            cleaned = clean_text(block["text"])
            if cleaned:
                lines.append(cleaned)
    return "\n\n".join(lines)


In [71]:
def table_to_text(table):
    heading = clean_text(table.get("heading", ""))

    rows = table.get("data", [])
    row_lines = []
    for row in rows:
        line = " | ".join(clean_text(str(cell)) for cell in row)
        row_lines.append(line)

    table_text = "\n".join(row_lines)
    return (heading + "\n" + table_text).strip()


In [72]:
class MiniLMEmbeddings:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts):
        # SentenceTransformer returns List[List[float]] when convert_to_numpy=False
        return self.model.encode(texts, convert_to_numpy=False)

    def embed_query(self, text):
        return self.model.encode([text], convert_to_numpy=False)[0]

In [73]:
embeddings = MiniLMEmbeddings(model_name="all-MiniLM-L6-v2")

semantic_chunker = SemanticChunker(embeddings)
semantic_chunker.chunk_size = 900
semantic_chunker.chunk_overlap = 150
semantic_chunker.separators = ["\n\n", "\n", " ", ""]

def chunk_text(text):
    docs = semantic_chunker.create_documents([text])
    return [doc.page_content for doc in docs]

In [74]:
def process_page(pdf_name, page):
    blocks, tables = load_page_components(pdf_name, page)

    block_text = merge_page_text(blocks)

    table_texts = [table_to_text(t) for t in tables]
    table_text = "\n\n".join(t for t in table_texts if t.strip())

    page_text = (block_text + "\n\n" + table_text).strip()
    
    if not page_text.strip():
        return []

    chunks = chunk_text(page_text)

    final = []
    for idx, chunk in enumerate(chunks):
        final.append({
            "text": chunk,
            "metadata": {
                "pdf_name": pdf_name,
                "page": page,
                "chunk_id": f"{pdf_name}_p{page}_c{idx}"
            }
        })
    return final

In [75]:
def process_pdf(pdf_name, page_count):
    all_chunks = []
    for p in range(1, page_count + 1):
        print(f"Processing page {p}/{page_count}")
        all_chunks.extend(process_page(pdf_name, p))
    return all_chunks

In [76]:
def save_jsonl(chunks, pdf_name):
    out_path = CHUNKED_DIR / f"{pdf_name}_semantic_chunks.jsonl"
    with open(out_path, "w", encoding="utf-8") as f:
        for c in chunks:
            f.write(json.dumps(c, ensure_ascii=False) + "\n")
    print("Saved:", out_path)

In [79]:
pdf_paths = list(PDF_DIR.glob("*.pdf"))

if not pdf_paths:
    print("No PDF files found in:", PDF_DIR)
else:
    print("PDFs detected:")
    for p in pdf_paths:
        print(" -", p.name)

for pdf in pdf_paths:
    pdf_name = pdf.stem
    print("\n==============================")
    print(f"Processing PDF: {pdf_name}")
    print("==============================")

    reader = PdfReader(str(pdf))
    page_count = len(reader.pages)
    print(f"Detected {page_count} pages.")

    chunks = process_pdf(pdf_name, page_count)

    save_jsonl(chunks, pdf_name)

print("\nAll PDFs processed!")


PDFs detected:
 - common childhood.pdf
 - Guidance for national.pdf
 - Latent.pdf
 - module1.pdf
 - module2.pdf
 - module3.pdf
 - module4.pdf
 - module5.pdf
 - module6.pdf
 - Recommendations.pdf

Processing PDF: common childhood
Detected 176 pages.
Processing page 1/176
Processing page 2/176
Processing page 3/176
Processing page 4/176
Processing page 5/176
Processing page 6/176
Processing page 7/176
Processing page 8/176
Processing page 9/176
Processing page 10/176
Processing page 11/176
Processing page 12/176
Processing page 13/176
Processing page 14/176
Processing page 15/176
Processing page 16/176
Processing page 17/176
Processing page 18/176
Processing page 19/176
Processing page 20/176
Processing page 21/176
Processing page 22/176
Processing page 23/176
Processing page 24/176
Processing page 25/176
Processing page 26/176
Processing page 27/176
Processing page 28/176
Processing page 29/176
Processing page 30/176
Processing page 31/176
Processing page 32/176
Processing page 33/176
P